In [1]:
import pandas as pd
import numpy as np
import re
import json
import ast
from pathlib import Path
from sklearn.decomposition import TruncatedSVD

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

# Project base directory
BASE_DIR = Path(r"C:\D drive\item_recommendation_model_create")

# Main data directory
DATA_DIR = BASE_DIR / "data"

# Separate output folders
CATEGORY_LABEL_OUTPUT_DIR = DATA_DIR / "category_label_output"
BASKET_EMBEDDING_OUTPUT_DIR = DATA_DIR / "basket_embedding_output"

# Create output folders automatically
CATEGORY_LABEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BASKET_EMBEDDING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Input file
INPUT_FILE = DATA_DIR / "main_data.csv"

# Output files
CATEGORY_LABEL_OUTPUT_FILE = CATEGORY_LABEL_OUTPUT_DIR / "customer_purchase_pattern_history_date_only.csv"
FINAL_OUTPUT_FILE = BASKET_EMBEDDING_OUTPUT_DIR / "customer_purchase_pattern_history_final.csv"
CATEGORY_EMBEDDING_LOOKUP_FILE = BASKET_EMBEDDING_OUTPUT_DIR / "category_embedding_lookup.csv"

print("Setup completed")
print("Input file:", INPUT_FILE)
print("Category label output:", CATEGORY_LABEL_OUTPUT_FILE)
print("Basket embedding output:", FINAL_OUTPUT_FILE)
print("Category embedding lookup:", CATEGORY_EMBEDDING_LOOKUP_FILE)

Setup completed
Input file: C:\D drive\item_recommendation_model_create\data\main_data.csv
Category label output: C:\D drive\item_recommendation_model_create\data\category_label_output\customer_purchase_pattern_history_date_only.csv
Basket embedding output: C:\D drive\item_recommendation_model_create\data\basket_embedding_output\customer_purchase_pattern_history_final.csv
Category embedding lookup: C:\D drive\item_recommendation_model_create\data\basket_embedding_output\category_embedding_lookup.csv


In [3]:
df = pd.read_csv(INPUT_FILE)

print("Loaded shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

Loaded shape: (1000000, 16)
Columns:
['transactionId', 'customerId', 'customerName', 'customerPersona', 'itemId', 'itemName', 'category', 'quantity', 'orderDate', 'isHoliday', 'isFestival', 'season', 'timeSlot', 'dayOfWeek', 'hour', 'month']


,transactionId,customerId,customerName,customerPersona,itemId,itemName,category,quantity,orderDate,isHoliday,isFestival,season,timeSlot,dayOfWeek,hour,month
0,1,23433,MD MOSSIN,family_grocery,952,Chashi Aroma.Chinigura Rice-2kg,Pantry-Rice,6,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
1,1,23433,MD MOSSIN,family_grocery,453,Saad Testy Bit Salt-100gm,pantry salt,1,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
2,1,23433,MD MOSSIN,family_grocery,15262,Cow Brain-K,Meat-Fresh,2,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
3,1,23433,MD MOSSIN,family_grocery,32441,Ela vista pomace olive oil 250 ml,Pantry-Oils,1,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
4,1,23433,MD MOSSIN,family_grocery,13986,Cow Stomach-K,Meat-Fresh,2,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1


In [4]:
required_columns = [
    "transactionId",
    "customerId",
    "itemId",
    "itemName",
    "category",
    "orderDate",
    "isHoliday",
    "isFestival",
    "season",
    "timeSlot"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df.copy()

df["orderDate"] = pd.to_datetime(df["orderDate"])
df["purchaseDate"] = df["orderDate"].dt.date

df["category"] = df["category"].astype(str).str.strip()
df["itemName"] = df["itemName"].astype(str).str.strip()
df["itemId"] = df["itemId"].astype(int)

print("Data validation completed")
display(df[required_columns + ["purchaseDate"]].head())

Data validation completed


,transactionId,customerId,itemId,itemName,category,orderDate,isHoliday,isFestival,season,timeSlot,purchaseDate
0,1,23433,952,Chashi Aroma.Chinigura Rice-2kg,Pantry-Rice,2025-01-01 15:40:00,0,1,Winter,Afternoon,2025-01-01
1,1,23433,453,Saad Testy Bit Salt-100gm,pantry salt,2025-01-01 15:40:00,0,1,Winter,Afternoon,2025-01-01
2,1,23433,15262,Cow Brain-K,Meat-Fresh,2025-01-01 15:40:00,0,1,Winter,Afternoon,2025-01-01
3,1,23433,32441,Ela vista pomace olive oil 250 ml,Pantry-Oils,2025-01-01 15:40:00,0,1,Winter,Afternoon,2025-01-01
4,1,23433,13986,Cow Stomach-K,Meat-Fresh,2025-01-01 15:40:00,0,1,Winter,Afternoon,2025-01-01


In [5]:
def get_month_part_label(dt):
    day = dt.day
    
    if day <= 10:
        return 1
    elif day <= 20:
        return 2
    else:
        return 3


def get_week_of_month(dt):
    first_day = dt.replace(day=1)
    adjusted_day = dt.day + first_day.weekday()
    return int(np.ceil(adjusted_day / 7.0))


def season_to_label(season):
    mapping = {
        "Winter": 1,
        "Summer": 2,
        "Rainy": 3
    }
    return mapping.get(str(season).strip(), 0)


def timeslot_to_label(timeslot):
    mapping = {
        "Morning": 1,
        "Noon": 2,
        "Afternoon": 3,
        "Evening": 4,
        "Night": 5
    }
    return mapping.get(str(timeslot).strip(), 0)


def to_curly_braces(values):
    clean_values = []
    
    for value in values:
        value = str(value).strip()
        
        if value and value.lower() != "nan":
            clean_values.append(value)
    
    unique_values = list(dict.fromkeys(clean_values))
    
    return "{" + ", ".join(unique_values) + "}"


print("Helper functions ready")

Helper functions ready


In [6]:
df["monthPartLabel"] = df["orderDate"].apply(get_month_part_label)
df["weekOfMonth"] = df["orderDate"].apply(get_week_of_month)
df["seasonLabel"] = df["season"].apply(season_to_label)
df["timeSlotLabel"] = df["timeSlot"].apply(timeslot_to_label)

display(
    df[
        [
            "transactionId",
            "customerId",
            "purchaseDate",
            "category",
            "itemId",
            "itemName",
            "season",
            "seasonLabel",
            "timeSlot",
            "timeSlotLabel",
            "monthPartLabel",
            "weekOfMonth"
        ]
    ].head()
)

,transactionId,customerId,purchaseDate,category,itemId,itemName,season,seasonLabel,timeSlot,timeSlotLabel,monthPartLabel,weekOfMonth
0,1,23433,2025-01-01,Pantry-Rice,952,Chashi Aroma.Chinigura Rice-2kg,Winter,1,Afternoon,3,1,1
1,1,23433,2025-01-01,pantry salt,453,Saad Testy Bit Salt-100gm,Winter,1,Afternoon,3,1,1
2,1,23433,2025-01-01,Meat-Fresh,15262,Cow Brain-K,Winter,1,Afternoon,3,1,1
3,1,23433,2025-01-01,Pantry-Oils,32441,Ela vista pomace olive oil 250 ml,Winter,1,Afternoon,3,1,1
4,1,23433,2025-01-01,Meat-Fresh,13986,Cow Stomach-K,Winter,1,Afternoon,3,1,1


In [7]:
customer_purchase_pattern_history = (
    df.sort_values(["customerId", "orderDate"])
      .groupby(["transactionId", "customerId", "purchaseDate"], as_index=False)
      .agg(
          categorySet=("category", to_curly_braces),
          itemIdSet=("itemId", to_curly_braces),
          itemNameSet=("itemName", to_curly_braces),
          season=("season", "first"),
          seasonLabel=("seasonLabel", "first"),
          isHoliday=("isHoliday", "first"),
          isFestival=("isFestival", "first"),
          timeSlot=("timeSlot", "first"),
          timeSlotLabel=("timeSlotLabel", "first"),
          monthPartLabel=("monthPartLabel", "first"),
          weekOfMonth=("weekOfMonth", "first")
      )
)

print("Basket level dataset shape:", customer_purchase_pattern_history.shape)
display(customer_purchase_pattern_history.head())

Basket level dataset shape: (184878, 14)


,transactionId,customerId,purchaseDate,categorySet,itemIdSet,itemNameSet,season,seasonLabel,isHoliday,isFestival,timeSlot,timeSlotLabel,monthPartLabel,weekOfMonth
0,1,23433,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, Pantry-Oils, Fish-Fresh, Spices-Cooking, Household-Laundry, Veg-Cooking}","{952, 453, 15262, 32441, 13986, 332, 13917, 13921, 704, 292, 14975}","{Chashi Aroma.Chinigura Rice-2kg, Saad Testy Bit Salt-100gm, Cow Brain-K, Ela vista pomace olive oil 250 ml, Cow Sto...",Winter,1,0,1,Afternoon,3,1,1
1,2,23553,2025-01-01,"{Meat-Fresh, Pantry-Oils, pantry salt, Pantry-Rice, Beverage-Juice}","{13986, 12655, 332, 952, 3191, 976}","{Cow Stomach-K, Ramisa Nut(Oil)-200gm, ACI Pure Salt-1Kg, Chashi Aroma.Chinigura Rice-2kg, Frooto Mango Juice-500ml,...",Winter,1,0,1,Afternoon,3,1,1
2,3,23453,2025-01-01,"{Desserts-Traditional, pantry salt, clothing accessories female, Dairy-Other, general_cooking_vegetables, clothing f...","{12637, 453, 32269, 17809, 6848, 25811}","{Ramisa-Nakshi Pitha Box-4 Pcs, Saad Testy Bit Salt-100gm, Gopal Ind Like Me Bra S-38 (4/23), Foster Clarks Corn Flo...",Winter,1,0,1,Noon,2,1,1
3,4,23437,2025-01-01,"{Pantry-Rice, Fish-Fresh, Veg-Cooking, pantry salt, Pantry-Pulses, Fruits-Fresh, Pantry-Sweeteners}","{2030, 13958, 14983, 453, 488, 15131, 976, 4958}","{Pran Aromatic Rice-2kg, Shrimp Fish Bagda, Cauliflower(B)-Pc, Saad Testy Bit Salt-100gm, Bacha Boot Pulse 1Kg, Atta...",Winter,1,0,1,Noon,2,1,1
4,5,23401,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, sauce item, Pantry-Grains, Beverage-Juice, Spices-Cooking}","{976, 332, 13986, 3612, 12826, 3191, 12834, 952}","{Lata Rice-KG, ACI Pure Salt-1Kg, Cow Stomach-K, Italiano Sweet Chilli Sauce-340g, Chia Seed (Dana) New-100g, Frooto...",Winter,1,0,1,Night,5,1,1


In [8]:
customer_purchase_pattern_history.to_csv(CATEGORY_LABEL_OUTPUT_FILE, index=False)

print("Saved:", CATEGORY_LABEL_OUTPUT_FILE)

print("\nSeason label distribution:")
print(customer_purchase_pattern_history["seasonLabel"].value_counts(dropna=False).sort_index())

print("\nTime slot label distribution:")
print(customer_purchase_pattern_history["timeSlotLabel"].value_counts(dropna=False).sort_index())

print("\nMonth part label distribution:")
print(customer_purchase_pattern_history["monthPartLabel"].value_counts(dropna=False).sort_index())

print("\nWeek of month distribution:")
print(customer_purchase_pattern_history["weekOfMonth"].value_counts(dropna=False).sort_index())

Saved: C:\D drive\item_recommendation_model_create\data\category_label_output\customer_purchase_pattern_history_date_only.csv

Season label distribution:
seasonLabel
1    56486
2    83564
3    44828
Name: count, dtype: int64

Time slot label distribution:
timeSlotLabel
1    29187
2    45437
3    32857
4    31651
5    45746
Name: count, dtype: int64

Month part label distribution:
monthPartLabel
1    63410
2    61761
3    59707
Name: count, dtype: int64

Week of month distribution:
weekOfMonth
1    24296
2    43974
3    43052
4    41146
5    30908
6     1502
Name: count, dtype: int64


In [9]:
def parse_curly_set(value):
    if pd.isna(value):
        return []
    
    value = str(value).strip()
    
    if value.startswith("{") and value.endswith("}"):
        value = value[1:-1]
    
    if value.strip() == "":
        return []
    
    return [x.strip() for x in value.split(",") if x.strip()]


embedding_df = customer_purchase_pattern_history.copy()

embedding_df["category_list"] = embedding_df["categorySet"].apply(parse_curly_set)

display(embedding_df[["customerId", "purchaseDate", "categorySet", "category_list"]].head())

,customerId,purchaseDate,categorySet,category_list
0,23433,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, Pantry-Oils, Fish-Fresh, Spices-Cooking, Household-Laundry, Veg-Cooking}","[Pantry-Rice, pantry salt, Meat-Fresh, Pantry-Oils, Fish-Fresh, Spices-Cooking, Household-Laundry, Veg-Cooking]"
1,23553,2025-01-01,"{Meat-Fresh, Pantry-Oils, pantry salt, Pantry-Rice, Beverage-Juice}","[Meat-Fresh, Pantry-Oils, pantry salt, Pantry-Rice, Beverage-Juice]"
2,23453,2025-01-01,"{Desserts-Traditional, pantry salt, clothing accessories female, Dairy-Other, general_cooking_vegetables, clothing f...","[Desserts-Traditional, pantry salt, clothing accessories female, Dairy-Other, general_cooking_vegetables, clothing f..."
3,23437,2025-01-01,"{Pantry-Rice, Fish-Fresh, Veg-Cooking, pantry salt, Pantry-Pulses, Fruits-Fresh, Pantry-Sweeteners}","[Pantry-Rice, Fish-Fresh, Veg-Cooking, pantry salt, Pantry-Pulses, Fruits-Fresh, Pantry-Sweeteners]"
4,23401,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, sauce item, Pantry-Grains, Beverage-Juice, Spices-Cooking}","[Pantry-Rice, pantry salt, Meat-Fresh, sauce item, Pantry-Grains, Beverage-Juice, Spices-Cooking]"


In [10]:
all_categories = sorted(
    {
        category
        for category_list in embedding_df["category_list"]
        for category in category_list
    }
)

print("Total unique categories:", len(all_categories))
print(all_categories)

Total unique categories: 46
['Bakery-Bread', 'Beverage-Carbonated', 'Beverage-Hot', 'Beverage-Juice', 'Beverage-Water', 'Chocolates-Sweets', 'Dairy-Milk', 'Dairy-Other', 'Desserts-Traditional', 'Dry-Fruits', 'Fish-Fresh', 'Fruits-Fresh', 'Household-AirCare', 'Household-Cleaning', 'Household-Kitchen', 'Household-Laundry', 'Household-Utility', 'Meat-Fresh', 'Meat-Processed', 'Pantry-Flour', 'Pantry-Grains', 'Pantry-Oils', 'Pantry-Pulses', 'Pantry-Rice', 'Pantry-Sweeteners', 'Personal-Care-Bath', 'Personal-Care-Cosmetics', 'Personal-Care-Hair', 'Personal-Care-Oral', 'Personal-Care-Sanitary', 'Pickle', 'Protein-Egg', 'Snacks-General', 'Spices-Cooking', 'Veg-Cooking', 'cleaning_clothes', 'clothing - male', 'clothing accessories female', 'clothing accessories male', 'clothing babies', 'clothing female', 'general_cooking_vegetables', 'household hygene', 'noodles_pasta_and_haleem', 'pantry salt', 'sauce item']


In [11]:
basket_category_matrix = pd.DataFrame(
    0,
    index=embedding_df.index,
    columns=all_categories,
    dtype=np.int8
)

for idx, category_list in embedding_df["category_list"].items():
    for category in category_list:
        if category in basket_category_matrix.columns:
            basket_category_matrix.loc[idx, category] = 1

print("Basket category matrix shape:", basket_category_matrix.shape)
display(basket_category_matrix.head())

Basket category matrix shape: (184878, 46)


,Bakery-Bread,Beverage-Carbonated,Beverage-Hot,Beverage-Juice,Beverage-Water,Chocolates-Sweets,Dairy-Milk,Dairy-Other,Desserts-Traditional,Dry-Fruits,Fish-Fresh,Fruits-Fresh,Household-AirCare,Household-Cleaning,Household-Kitchen,Household-Laundry,Household-Utility,Meat-Fresh,Meat-Processed,Pantry-Flour,Pantry-Grains,Pantry-Oils,Pantry-Pulses,Pantry-Rice,Pantry-Sweeteners,Personal-Care-Bath,Personal-Care-Cosmetics,Personal-Care-Hair,Personal-Care-Oral,Personal-Care-Sanitary,Pickle,Protein-Egg,Snacks-General,Spices-Cooking,Veg-Cooking,cleaning_clothes,clothing - male,clothing accessories female,clothing accessories male,clothing babies,clothing female,general_cooking_vegetables,household hygene,noodles_pasta_and_haleem,pantry salt,sauce item
0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0
1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0
3,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0
4,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1


In [12]:
category_cooccurrence_matrix = basket_category_matrix.T.dot(basket_category_matrix)

print("Category cooccurrence matrix shape:", category_cooccurrence_matrix.shape)
display(category_cooccurrence_matrix.head())

Category cooccurrence matrix shape: (46, 46)


,Bakery-Bread,Beverage-Carbonated,Beverage-Hot,Beverage-Juice,Beverage-Water,Chocolates-Sweets,Dairy-Milk,Dairy-Other,Desserts-Traditional,Dry-Fruits,Fish-Fresh,Fruits-Fresh,Household-AirCare,Household-Cleaning,Household-Kitchen,Household-Laundry,Household-Utility,Meat-Fresh,Meat-Processed,Pantry-Flour,Pantry-Grains,Pantry-Oils,Pantry-Pulses,Pantry-Rice,Pantry-Sweeteners,Personal-Care-Bath,Personal-Care-Cosmetics,Personal-Care-Hair,Personal-Care-Oral,Personal-Care-Sanitary,Pickle,Protein-Egg,Snacks-General,Spices-Cooking,Veg-Cooking,cleaning_clothes,clothing - male,clothing accessories female,clothing accessories male,clothing babies,clothing female,general_cooking_vegetables,household hygene,noodles_pasta_and_haleem,pantry salt,sauce item
Bakery-Bread,25,75,-12,71,44,-60,124,30,-105,-40,-124,-70,10,48,112,34,-7,47,96,21,20,-109,41,51,48,74,-31,-55,-72,125,-100,126,124,74,-9,20,-119,2,-113,-38,-48,-124,18,93,17,65
Beverage-Carbonated,75,24,24,86,-68,48,31,115,126,-87,68,-6,-71,5,-4,37,-19,-125,-40,86,-43,-95,102,-125,112,-9,-74,-39,-118,-113,-42,-85,-88,120,74,-99,-24,89,124,124,94,62,113,95,118,-75
Beverage-Hot,-12,24,-91,59,50,50,-79,-6,43,27,-126,-21,-21,107,72,56,82,79,113,-4,-8,-71,66,33,-20,27,44,-18,85,83,-11,-28,62,-128,118,-70,-92,-32,102,72,2,75,72,-82,-128,-35
Beverage-Juice,71,86,59,61,-29,-123,109,-66,-10,54,22,67,101,-116,79,116,75,-94,-55,91,71,88,33,104,107,-57,-117,-80,88,56,11,77,-84,102,1,61,88,104,-101,-65,-54,111,-87,47,76,-104
Beverage-Water,44,-68,50,-29,-47,86,-27,-15,-25,63,122,82,86,103,42,120,115,-72,36,-2,55,15,-78,-66,-74,-115,70,-112,-34,-100,78,20,61,13,43,69,122,37,-41,-105,-127,16,-61,25,-1,30


In [13]:
n_categories = len(all_categories)

if n_categories <= 2:
    embedding_dim = 2
else:
    embedding_dim = min(8, n_categories - 1)

print("Embedding dimension:", embedding_dim)

svd = TruncatedSVD(n_components=embedding_dim, random_state=42)

category_embeddings = svd.fit_transform(category_cooccurrence_matrix)

category_embedding_df = pd.DataFrame(
    category_embeddings,
    index=all_categories,
    columns=[f"cat_emb_{i+1}" for i in range(embedding_dim)]
)

category_embedding_df = category_embedding_df.reset_index().rename(columns={"index": "category"})

print("Explained variance ratio sum:", svd.explained_variance_ratio_.sum())
display(category_embedding_df.head())

Embedding dimension: 8
Explained variance ratio sum: 0.50392291934286


,category,cat_emb_1,cat_emb_2,cat_emb_3,cat_emb_4,cat_emb_5,cat_emb_6,cat_emb_7,cat_emb_8
0,Bakery-Bread,-74.173568,-111.558025,-192.203051,-70.900538,-27.881714,-73.214051,24.189736,161.398282
1,Beverage-Carbonated,-85.186907,-42.566291,242.853551,7.961231,-155.819533,0.959077,-104.045821,152.333994
2,Beverage-Hot,-28.961506,-26.255853,-7.816855,264.783572,116.117895,19.741289,-79.332576,-52.988355
3,Beverage-Juice,121.029090,179.488928,-119.588266,-205.621190,-16.090835,-78.735413,-165.299904,179.731694
4,Beverage-Water,284.556475,104.078329,-57.684418,9.785665,-68.273613,-132.973486,-38.705386,-119.340987


In [14]:
category_to_vector = {
    row["category"]: row[[f"cat_emb_{i+1}" for i in range(embedding_dim)]].values.astype(float)
    for _, row in category_embedding_df.iterrows()
}


def create_basket_category_embedding(category_list):
    vectors = []
    
    for category in category_list:
        if category in category_to_vector:
            vectors.append(category_to_vector[category])
    
    if len(vectors) == 0:
        return [0.0] * embedding_dim
    
    basket_vector = np.mean(vectors, axis=0)
    
    return basket_vector.round(6).tolist()


embedding_df["basket_category_embedding"] = embedding_df["category_list"].apply(create_basket_category_embedding)

display(
    embedding_df[
        [
            "customerId",
            "purchaseDate",
            "categorySet",
            "basket_category_embedding"
        ]
    ].head()
)

,customerId,purchaseDate,categorySet,basket_category_embedding
0,23433,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, Pantry-Oils, Fish-Fresh, Spices-Cooking, Household-Laundry, Veg-Cooking}","[10.822356, -72.828976, -67.03191, 3.619922, -10.427063, -31.310562, -19.462887, 57.080152]"
1,23553,2025-01-01,"{Meat-Fresh, Pantry-Oils, pantry salt, Pantry-Rice, Beverage-Juice}","[-4.513359, 25.17083, -71.078044, -62.388689, 20.547252, -60.075294, -38.494798, 81.975637]"
2,23453,2025-01-01,"{Desserts-Traditional, pantry salt, clothing accessories female, Dairy-Other, general_cooking_vegetables, clothing f...","[-7.021769, 46.920831, 104.087329, 39.232723, 19.593457, -8.814851, -31.844793, -5.327298]"
3,23437,2025-01-01,"{Pantry-Rice, Fish-Fresh, Veg-Cooking, pantry salt, Pantry-Pulses, Fruits-Fresh, Pantry-Sweeteners}","[-22.928417, -20.921959, -17.457138, 86.736529, 23.317877, 24.226424, -44.764845, 29.782153]"
4,23401,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, sauce item, Pantry-Grains, Beverage-Juice, Spices-Cooking}","[36.004871, 45.367819, -81.582129, 38.222005, 20.15793, -29.854635, -51.6094, 73.25617]"


In [15]:
final_df = embedding_df.drop(columns=["category_list"], errors="ignore").copy()

final_df.to_csv(FINAL_OUTPUT_FILE, index=False)
category_embedding_df.to_csv(CATEGORY_EMBEDDING_LOOKUP_FILE, index=False)

print("Saved:", FINAL_OUTPUT_FILE)
print("Saved:", CATEGORY_EMBEDDING_LOOKUP_FILE)

print("Final shape:", final_df.shape)
display(final_df.head())

Saved: C:\D drive\item_recommendation_model_create\data\basket_embedding_output\customer_purchase_pattern_history_final.csv
Saved: C:\D drive\item_recommendation_model_create\data\basket_embedding_output\category_embedding_lookup.csv
Final shape: (184878, 15)


,transactionId,customerId,purchaseDate,categorySet,itemIdSet,itemNameSet,season,seasonLabel,isHoliday,isFestival,timeSlot,timeSlotLabel,monthPartLabel,weekOfMonth,basket_category_embedding
0,1,23433,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, Pantry-Oils, Fish-Fresh, Spices-Cooking, Household-Laundry, Veg-Cooking}","{952, 453, 15262, 32441, 13986, 332, 13917, 13921, 704, 292, 14975}","{Chashi Aroma.Chinigura Rice-2kg, Saad Testy Bit Salt-100gm, Cow Brain-K, Ela vista pomace olive oil 250 ml, Cow Sto...",Winter,1,0,1,Afternoon,3,1,1,"[10.822356, -72.828976, -67.03191, 3.619922, -10.427063, -31.310562, -19.462887, 57.080152]"
1,2,23553,2025-01-01,"{Meat-Fresh, Pantry-Oils, pantry salt, Pantry-Rice, Beverage-Juice}","{13986, 12655, 332, 952, 3191, 976}","{Cow Stomach-K, Ramisa Nut(Oil)-200gm, ACI Pure Salt-1Kg, Chashi Aroma.Chinigura Rice-2kg, Frooto Mango Juice-500ml,...",Winter,1,0,1,Afternoon,3,1,1,"[-4.513359, 25.17083, -71.078044, -62.388689, 20.547252, -60.075294, -38.494798, 81.975637]"
2,3,23453,2025-01-01,"{Desserts-Traditional, pantry salt, clothing accessories female, Dairy-Other, general_cooking_vegetables, clothing f...","{12637, 453, 32269, 17809, 6848, 25811}","{Ramisa-Nakshi Pitha Box-4 Pcs, Saad Testy Bit Salt-100gm, Gopal Ind Like Me Bra S-38 (4/23), Foster Clarks Corn Flo...",Winter,1,0,1,Noon,2,1,1,"[-7.021769, 46.920831, 104.087329, 39.232723, 19.593457, -8.814851, -31.844793, -5.327298]"
3,4,23437,2025-01-01,"{Pantry-Rice, Fish-Fresh, Veg-Cooking, pantry salt, Pantry-Pulses, Fruits-Fresh, Pantry-Sweeteners}","{2030, 13958, 14983, 453, 488, 15131, 976, 4958}","{Pran Aromatic Rice-2kg, Shrimp Fish Bagda, Cauliflower(B)-Pc, Saad Testy Bit Salt-100gm, Bacha Boot Pulse 1Kg, Atta...",Winter,1,0,1,Noon,2,1,1,"[-22.928417, -20.921959, -17.457138, 86.736529, 23.317877, 24.226424, -44.764845, 29.782153]"
4,5,23401,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, sauce item, Pantry-Grains, Beverage-Juice, Spices-Cooking}","{976, 332, 13986, 3612, 12826, 3191, 12834, 952}","{Lata Rice-KG, ACI Pure Salt-1Kg, Cow Stomach-K, Italiano Sweet Chilli Sauce-340g, Chia Seed (Dana) New-100g, Frooto...",Winter,1,0,1,Night,5,1,1,"[36.004871, 45.367819, -81.582129, 38.222005, 20.15793, -29.854635, -51.6094, 73.25617]"


In [16]:
print("Final columns:")
print(final_df.columns.tolist())

print("\nFinal sample:")
display(final_df.head(10))

print("\nEmbedding sample:")
display(final_df[["customerId", "purchaseDate", "categorySet", "basket_category_embedding"]].head(10))

Final columns:
['transactionId', 'customerId', 'purchaseDate', 'categorySet', 'itemIdSet', 'itemNameSet', 'season', 'seasonLabel', 'isHoliday', 'isFestival', 'timeSlot', 'timeSlotLabel', 'monthPartLabel', 'weekOfMonth', 'basket_category_embedding']

Final sample:


,transactionId,customerId,purchaseDate,categorySet,itemIdSet,itemNameSet,season,seasonLabel,isHoliday,isFestival,timeSlot,timeSlotLabel,monthPartLabel,weekOfMonth,basket_category_embedding
0,1,23433,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, Pantry-Oils, Fish-Fresh, Spices-Cooking, Household-Laundry, Veg-Cooking}","{952, 453, 15262, 32441, 13986, 332, 13917, 13921, 704, 292, 14975}","{Chashi Aroma.Chinigura Rice-2kg, Saad Testy Bit Salt-100gm, Cow Brain-K, Ela vista pomace olive oil 250 ml, Cow Sto...",Winter,1,0,1,Afternoon,3,1,1,"[10.822356, -72.828976, -67.03191, 3.619922, -10.427063, -31.310562, -19.462887, 57.080152]"
1,2,23553,2025-01-01,"{Meat-Fresh, Pantry-Oils, pantry salt, Pantry-Rice, Beverage-Juice}","{13986, 12655, 332, 952, 3191, 976}","{Cow Stomach-K, Ramisa Nut(Oil)-200gm, ACI Pure Salt-1Kg, Chashi Aroma.Chinigura Rice-2kg, Frooto Mango Juice-500ml,...",Winter,1,0,1,Afternoon,3,1,1,"[-4.513359, 25.17083, -71.078044, -62.388689, 20.547252, -60.075294, -38.494798, 81.975637]"
2,3,23453,2025-01-01,"{Desserts-Traditional, pantry salt, clothing accessories female, Dairy-Other, general_cooking_vegetables, clothing f...","{12637, 453, 32269, 17809, 6848, 25811}","{Ramisa-Nakshi Pitha Box-4 Pcs, Saad Testy Bit Salt-100gm, Gopal Ind Like Me Bra S-38 (4/23), Foster Clarks Corn Flo...",Winter,1,0,1,Noon,2,1,1,"[-7.021769, 46.920831, 104.087329, 39.232723, 19.593457, -8.814851, -31.844793, -5.327298]"
3,4,23437,2025-01-01,"{Pantry-Rice, Fish-Fresh, Veg-Cooking, pantry salt, Pantry-Pulses, Fruits-Fresh, Pantry-Sweeteners}","{2030, 13958, 14983, 453, 488, 15131, 976, 4958}","{Pran Aromatic Rice-2kg, Shrimp Fish Bagda, Cauliflower(B)-Pc, Saad Testy Bit Salt-100gm, Bacha Boot Pulse 1Kg, Atta...",Winter,1,0,1,Noon,2,1,1,"[-22.928417, -20.921959, -17.457138, 86.736529, 23.317877, 24.226424, -44.764845, 29.782153]"
4,5,23401,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, sauce item, Pantry-Grains, Beverage-Juice, Spices-Cooking}","{976, 332, 13986, 3612, 12826, 3191, 12834, 952}","{Lata Rice-KG, ACI Pure Salt-1Kg, Cow Stomach-K, Italiano Sweet Chilli Sauce-340g, Chia Seed (Dana) New-100g, Frooto...",Winter,1,0,1,Night,5,1,1,"[36.004871, 45.367819, -81.582129, 38.222005, 20.15793, -29.854635, -51.6094, 73.25617]"
5,6,23432,2025-01-01,"{Beverage-Carbonated, Chocolates-Sweets, Bakery-Bread, Dry-Fruits, Pantry-Sweeteners}","{22259, 1217, 22235, 13654, 12963, 4958}","{Mirinda Orange- 600ml, Chupa Chups lolipop, Coca Cola-2.25Lt(Pet), Quality Premiun Paratha 10Pcs-650g, White Kazu B...",Winter,1,0,1,Noon,2,1,1,"[-36.402973, -59.149879, 78.049234, -53.018434, 1.394624, 67.807748, -19.2257, 73.503859]"
6,7,23479,2025-01-01,"{household hygene, Bakery-Bread, Protein-Egg, Beverage-Carbonated}","{25524, 13348, 14034, 22237}","{Dettol Handwash Aloe Vera - 200ml, Pandan Kaya Bread- 225g, Omega 3 Enriched Egg-12pcs, Coca Cola-600ml(Pet)}",Winter,1,0,1,Morning,1,1,1,"[-46.90429, 23.287991, -33.705841, -1.126547, -80.633039, -37.247385, -35.595805, 91.984257]"
7,8,18557,2025-01-01,"{Spices-Cooking, Protein-Egg, Meat-Fresh, Pantry-Rice, Fish-Fresh, Beverage-Juice}","{551, 14034, 15262, 2030, 12823, 14045, 952, 2801}","{Cumin Powder-100gm, Omega 3 Enriched Egg-12pcs, Cow Brain-K, Pran Aromatic Rice-2kg, Jafran, Pabda Fish (Big), Chas...",Winter,1,0,1,Noon,2,1,1,"[-30.478946, 17.626679, -133.201618, 33.294928, -3.350005, -42.779786, -17.799377, 51.929757]"
8,9,23403,2025-01-01,"{Desserts-Traditional, Dairy-Other, Pantry-Pulses, Beverage-Juice, Spices-Cooking, Pantry-Sweeteners}","{12637, 7182, 487, 3179, 604, 6838, 15180}","{Ramisa-Nakshi Pitha Box-4 Pcs, Aarong Cheese - 100gm, Prince Dubli Pulse-1kg, Starship Mango Juice 250ml pack, Cass...",Winter,1,0,1,Evening,4,1,1,"[-29.279188, 93.263698, 4.804092, 12.18242, -48.212961, 47.048816, -34.714325, 95.787462]"
9,10,23565,2025-01-01,"{Pantry-Flour, Dairy-Other, Beverage-Carbonated}","{30626, 7182, 22234}","{Bashundhara Maida-2kg, Aarong Cheese - 100gm, Sprite-1.25 Lt(Pet)}",Winter,1,0,1,Mo


Embedding sample:


,customerId,purchaseDate,categorySet,basket_category_embedding
0,23433,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, Pantry-Oils, Fish-Fresh, Spices-Cooking, Household-Laundry, Veg-Cooking}","[10.822356, -72.828976, -67.03191, 3.619922, -10.427063, -31.310562, -19.462887, 57.080152]"
1,23553,2025-01-01,"{Meat-Fresh, Pantry-Oils, pantry salt, Pantry-Rice, Beverage-Juice}","[-4.513359, 25.17083, -71.078044, -62.388689, 20.547252, -60.075294, -38.494798, 81.975637]"
2,23453,2025-01-01,"{Desserts-Traditional, pantry salt, clothing accessories female, Dairy-Other, general_cooking_vegetables, clothing f...","[-7.021769, 46.920831, 104.087329, 39.232723, 19.593457, -8.814851, -31.844793, -5.327298]"
3,23437,2025-01-01,"{Pantry-Rice, Fish-Fresh, Veg-Cooking, pantry salt, Pantry-Pulses, Fruits-Fresh, Pantry-Sweeteners}","[-22.928417, -20.921959, -17.457138, 86.736529, 23.317877, 24.226424, -44.764845, 29.782153]"
4,23401,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, sauce item, Pantry-Grains, Beverage-Juice, Spices-Cooking}","[36.004871, 45.367819, -81.582129, 38.222005, 20.15793, -29.854635, -51.6094, 73.25617]"
5,23432,2025-01-01,"{Beverage-Carbonated, Chocolates-Sweets, Bakery-Bread, Dry-Fruits, Pantry-Sweeteners}","[-36.402973, -59.149879, 78.049234, -53.018434, 1.394624, 67.807748, -19.2257, 73.503859]"
6,23479,2025-01-01,"{household hygene, Bakery-Bread, Protein-Egg, Beverage-Carbonated}","[-46.90429, 23.287991, -33.705841, -1.126547, -80.633039, -37.247385, -35.595805, 91.984257]"
7,18557,2025-01-01,"{Spices-Cooking, Protein-Egg, Meat-Fresh, Pantry-Rice, Fish-Fresh, Beverage-Juice}","[-30.478946, 17.626679, -133.201618, 33.294928, -3.350005, -42.779786, -17.799377, 51.929757]"
8,23403,2025-01-01,"{Desserts-Traditional, Dairy-Other, Pantry-Pulses, Beverage-Juice, Spices-Cooking, Pantry-Sweeteners}","[-29.279188, 93.263698, 4.804092, 12.18242, -48.212961, 47.048816, -34.714325, 95.787462]"
9,23565,2025-01-01,"{Pantry-Flour, Dairy-Other, Beverage-Carbonated}","[7.096231, 44.615265, 148.230392, 47.576116, -153.427743, 27.718074, -11.310347, 96.152558]"


In [17]:
check_df = pd.read_csv(FINAL_OUTPUT_FILE)

print("Reloaded final file shape:", check_df.shape)
print("Columns:")
print(check_df.columns.tolist())

display(check_df.head())

Reloaded final file shape: (184878, 15)
Columns:
['transactionId', 'customerId', 'purchaseDate', 'categorySet', 'itemIdSet', 'itemNameSet', 'season', 'seasonLabel', 'isHoliday', 'isFestival', 'timeSlot', 'timeSlotLabel', 'monthPartLabel', 'weekOfMonth', 'basket_category_embedding']


,transactionId,customerId,purchaseDate,categorySet,itemIdSet,itemNameSet,season,seasonLabel,isHoliday,isFestival,timeSlot,timeSlotLabel,monthPartLabel,weekOfMonth,basket_category_embedding
0,1,23433,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, Pantry-Oils, Fish-Fresh, Spices-Cooking, Household-Laundry, Veg-Cooking}","{952, 453, 15262, 32441, 13986, 332, 13917, 13921, 704, 292, 14975}","{Chashi Aroma.Chinigura Rice-2kg, Saad Testy Bit Salt-100gm, Cow Brain-K, Ela vista pomace olive oil 250 ml, Cow Sto...",Winter,1,0,1,Afternoon,3,1,1,"[10.822356, -72.828976, -67.03191, 3.619922, -10.427063, -31.310562, -19.462887, 57.080152]"
1,2,23553,2025-01-01,"{Meat-Fresh, Pantry-Oils, pantry salt, Pantry-Rice, Beverage-Juice}","{13986, 12655, 332, 952, 3191, 976}","{Cow Stomach-K, Ramisa Nut(Oil)-200gm, ACI Pure Salt-1Kg, Chashi Aroma.Chinigura Rice-2kg, Frooto Mango Juice-500ml,...",Winter,1,0,1,Afternoon,3,1,1,"[-4.513359, 25.17083, -71.078044, -62.388689, 20.547252, -60.075294, -38.494798, 81.975637]"
2,3,23453,2025-01-01,"{Desserts-Traditional, pantry salt, clothing accessories female, Dairy-Other, general_cooking_vegetables, clothing f...","{12637, 453, 32269, 17809, 6848, 25811}","{Ramisa-Nakshi Pitha Box-4 Pcs, Saad Testy Bit Salt-100gm, Gopal Ind Like Me Bra S-38 (4/23), Foster Clarks Corn Flo...",Winter,1,0,1,Noon,2,1,1,"[-7.021769, 46.920831, 104.087329, 39.232723, 19.593457, -8.814851, -31.844793, -5.327298]"
3,4,23437,2025-01-01,"{Pantry-Rice, Fish-Fresh, Veg-Cooking, pantry salt, Pantry-Pulses, Fruits-Fresh, Pantry-Sweeteners}","{2030, 13958, 14983, 453, 488, 15131, 976, 4958}","{Pran Aromatic Rice-2kg, Shrimp Fish Bagda, Cauliflower(B)-Pc, Saad Testy Bit Salt-100gm, Bacha Boot Pulse 1Kg, Atta...",Winter,1,0,1,Noon,2,1,1,"[-22.928417, -20.921959, -17.457138, 86.736529, 23.317877, 24.226424, -44.764845, 29.782153]"
4,5,23401,2025-01-01,"{Pantry-Rice, pantry salt, Meat-Fresh, sauce item, Pantry-Grains, Beverage-Juice, Spices-Cooking}","{976, 332, 13986, 3612, 12826, 3191, 12834, 952}","{Lata Rice-KG, ACI Pure Salt-1Kg, Cow Stomach-K, Italiano Sweet Chilli Sauce-340g, Chia Seed (Dana) New-100g, Frooto...",Winter,1,0,1,Night,5,1,1,"[36.004871, 45.367819, -81.582129, 38.222005, 20.15793, -29.854635, -51.6094, 73.25617]"
